# Endometriosis segmentation

Este notebook descreve o processo desenvolvido para realizar segmentacao de endometriose em imagens de ressonancia magnetica utilizando Deep Learning.

Este trabalho é minha interpretação do artigo:
https://www.nature.com/articles/s41597-025-05623-3

Dataset:
https://zenodo.org/records/15750762

Este codigo, junto com outros de utilidade e manipulação do dataset podem ser acessados em:
https://github.com/VSundermann/IMScience_endometriosisRecognition


## Objetivos da Pesquisa

Pergunta: é possivel delinear endometriose em exames de imagem medica?

Hipotese: Desenvolver um pipeline de rede neural capaz de segmentar lesões de endometriose em imagens de ressonancia magnetica


## Ambiente de execução e dependencias

Este notebook foi executado utilizando o Google Colab, o ambiente de execucao utilizado é o mais recente, contando com Python 3.12, Numpy 2.02.

Para criação dos modelos de Deep Learning utilizamos o framework Monai:
https://github.com/Project-MONAI/MONAI

O treinamento dos modelos foi realizado com GPUs A100, acessadas atraves da versao Pro do Google Colab, em modo de ALTA memoria RAM.


In [1]:
!python -c "import monai" || pip install -q "monai-weekly[tqdm, nibabel, gdown, ignite]"

!python -c "import SimpleITK" || pip install SimpleITK

#!pip install tensorboard-plugin-3d

Traceback (most recent call last):
  File "<string>", line 1, in <module>
ModuleNotFoundError: No module named 'monai'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.5/266.5 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 7.2 MB/s eta 0:00:00
Traceback (most recent call last):
  File "<string>", line 1, in <module>
ModuleNotFoundError: No module named 'SimpleITK'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 45.1 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Importação de dependendias e do codigo RAovSeg_tools, desenvolvido pelos autores do artigo mencionado, disponibilizado no meu github, que possui implementações das seguintes utilidades:

*   Normalização de imagens
*   Reshape de imagens
*   Pré-processamento
*   Função de perda - Focal Tversky
*   Metrica de Avaliação - Dice Similarity Coefficient
*   Pos-processamento - operações de closing e seleção do maior componente conexo


In [3]:
import sys
sys.path.insert(0, '/content/drive/MyDrive/Colab Notebooks/')
import RAovSeg_tools as tools

In [4]:
import SimpleITK as sitk
from ipywidgets import interact, interactive

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import random_split
from torch.utils.tensorboard import SummaryWriter

from datetime import datetime

from glob import glob
import os
import io
import shutil

In [5]:
import monai
from monai.data import decollate_batch, DataLoader, ImageDataset
from monai.visualize import plot_2d_or_3d_image, blend_images
from monai.inferers import sliding_window_inference
from monai.metrics import DiceMetric, ConfusionMatrixMetric, get_confusion_matrix
from monai.losses import DiceLoss
from monai.transforms import (
    Activations,
    AsDiscrete,
    Compose,
    DivisiblePad,
    EnsureChannelFirst,
    LoadImage,
    ScaleIntensity,
    SaveImage
)

# Dataset

O dataset utilizado neste trabalho foi coletado em duas instutuições, Memorial Hermann Hospital System e Texas Children’s Hospital Pavilion for Women.

As imagens possuem anotacoes de medicos que determinam se foi localizado os seguintes orgãos ou lesoes:

*   ut - utero
*   ov - ovario
*   cy - cisto
*   em - endomtrioma
*   cds - Cul de Sac (não ha imagens com esta anotação)

As imagens de ressonancia podem ser dos seguintes tipos:

*   T1 - realça líquidos e gorduras, brilhantes
*   T1FS - "" com supressão de gordura
*   T2 - líquidos - escuros, gorduras - mais claras
*   T2FS - "" com supressão de gordura
*   PAT - patient file, pode ser qualquer um dos tipos acima

Portanto, no dataset as imagens são nomeadas da seguinte forma
DX_YYY_ZZ.nii.gz

X = Instituição coletada(1 - MHHS, 2 - TCHPfW)

Y = Identificação do paciente(varia de 000 até 080)

Z = Corresponde ao tipo da ressonancia magnetica ou à anotação realizada

## Pré-processamento do dataset

Como descrito do bloco anterior, as imagens de RM e suas respectivas anotações estão misturadas, dessa forma, torna-se necessario organizar as imagens em diretorios especificos para cada tipo de anotação.

O codigo que realiza a organizacao do dataset, denominado dataset_prep.py pode ser acessado no meu github mencionado acima.

Organizacao do dataset
- mri_full
- mri_filtered
- label_full
- label_filtered
  - T1
  - T2
  - T1FS
  - T2FS

In [6]:
##########################################
### DATASET SETUP
##########################################

main_directory = '/content/drive/MyDrive/ImScience/dataset_resampled'
MRI_images_dir = os.path.join(main_directory, 'mri_filtered_slices/T2FS')
labels_dir = os.path.join(main_directory, 'label_filtered_slices/T2FS')

images = glob(os.path.join(MRI_images_dir, 'D2*.nii.gz'))
labels = glob(os.path.join(labels_dir, 'D2*.nii.gz'))

images.sort()
labels.sort()

## Visualização dos dados

In [ ]:
def interactive_exam_visualization(img_no, img_slice):
  patient_img = os.path.basename(images[img_no])
  patient_label = os.path.basename(labels[img_no])

  original_image = sitk.ReadImage(images[img_no])
  original_label = sitk.ReadImage(labels[img_no])

  img_array = sitk.GetArrayFromImage(original_image)[img_slice, :, :]
  lbl_array = sitk.GetArrayFromImage(original_label)[img_slice, :, :]

  masked_data = np.ma.masked_where(lbl_array == 0, lbl_array)

  plt.figure("check", (18, 6))
  plt.subplot(1, 3, 1)
  plt.title(f"Original Image" + patient_img)
  plt.imshow(img_array, cmap='gray')
  plt.subplot(1, 3, 2)
  plt.title(f"Original GT" + patient_label)
  plt.imshow(lbl_array, cmap='gray')
  plt.subplot(1, 3, 3)
  plt.title(f"Overlay")
  plt.imshow(img_array, cmap='gray')
  plt.imshow(masked_data, cmap='viridis', alpha=0.8)


interactive(interactive_exam_visualization, img_no=(0,len(images)), img_slice=(0,47))

interactive(children=(IntSlider(value=20, description='img_no', max=40), IntSlider(value=23, description='img_…

In [ ]:
def interactive_slices_visualization(slice_no):
  patient_img = os.path.basename(images[slice_no])
  patient_label = os.path.basename(labels[slice_no])

  original_image = sitk.ReadImage(images[slice_no])
  original_label = sitk.ReadImage(labels[slice_no])

  img_array = sitk.GetArrayFromImage(original_image)
  lbl_array = sitk.GetArrayFromImage(original_label)

  masked_data = np.ma.masked_where(lbl_array == 0, lbl_array)

  plt.figure("check", (18, 6))
  plt.subplot(1, 3, 1)
  plt.title(f"Original Image" + patient_img)
  plt.imshow(img_array, cmap='gray')
  plt.subplot(1, 3, 2)
  plt.title(f"Original GT" + patient_label)
  plt.imshow(lbl_array, cmap='gray')
  plt.subplot(1, 3, 3)
  plt.title(f"Overlay")
  plt.imshow(img_array, cmap='gray')
  plt.imshow(masked_data, cmap='viridis', alpha=0.8)

interactive(interactive_slices_visualization, slice_no=(0,len(images)-1))

interactive(children=(IntSlider(value=66, description='slice_no', max=132), Output()), _dom_classes=('widget-i…

## Divisão do dataset

Nesta seção dividimos o dataset em partes para treino, validação e teste, é possivel adicionar transformadas para realizar data augmentation.

Utiliza-se uma seed predefinida para garantir reprodutibilidade dos experimentos.

In [7]:
# Dataset utilizando a classe ImageDataset
# Responsavel por tratar metadados das imagens
class TestCompose(Compose):
    def __call__(self, data, meta, **_kwargs):
        data = self.transforms[0](data)  # ensure channel first
        data = self.transforms[1](data)  # spacing
        meta = data.meta
        if len(self.transforms) == 3:
            return self.transforms[2](data), meta  # image contrast
        return data, meta

img_trans = TestCompose([
    EnsureChannelFirst(),
    ScaleIntensity(),
    DivisiblePad(k=32),
])

label_trans = TestCompose([
    EnsureChannelFirst(),
    ScaleIntensity(),
    DivisiblePad(k=32),
])

img_dataset = ImageDataset(
    image_files=images,
    seg_files=labels,
    transform=img_trans,
    seg_transform=label_trans,
    image_only=False,
    transform_with_metadata=True,
)

train_partition = 0.8
val_partition = 0.1
test_partition = 0.1

total_size = len(img_dataset)
train_size = int(train_partition * total_size)
val_size = int(val_partition * total_size)
test_size = total_size - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(img_dataset, [train_size, val_size, test_size], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

dice_metric = DiceMetric(include_background=True, reduction="mean", get_not_nans=False)
post_trans = Compose([Activations(sigmoid=True), AsDiscrete(threshold=0.5)])
saver = SaveImage(output_dir="/content/drive/MyDrive/Colab Notebooks/EndoSeg_out", output_ext=".png", output_postfix="seg", scale=255)

print(f"Train set size: {len(train_dataset)}")
print(f"Validation set size: {len(val_dataset)}")
print(f"Test set size: {len(test_dataset)}")
print(f"Shape of images: {train_dataset[0][0].shape}")

Train set size: 47
Validation set size: 5
Test set size: 7
Shape of images: torch.Size([1, 512, 512])


# Arquitetura dos modelos

Nesta seção definimos os modelos que serão utilizados no trabalho, o artigo de referencia utiliza um classificador e um modelo de segmentação, no momento recriamos somente a tarefa de segmentação para facilitar o desenvolvimento.

Um classificador pode ser realizado para reconhecer o tipo de ressonancia magnetica de entrada e chamar um modelo de segmentação especializado para aquele tipo, mas não é o objetivo atual.

Selecionar um dos modelos listados abaixo.

In [8]:
##########################################
### MODEL PIPELINE
##########################################

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# @title
# ResClass
# It was trained on 2D MRI slices from all training subjects, utilizing 3,252 slices for training and 2,168 slices for validation.
# The model architecture is a two-layer 2D ResNet18 with 8 and 16 features in the respective layers.
# Binary Cross Entropy with Logits Loss (BCEWithLogitsLoss) was used to train the classifier

classification_model = monai.networks.nets.ResNetFeatures(model_name="resnet18").to(device)

resclass_loss = torch.nn.BCEWithLogitsLoss()

optimizer_classification = torch.optim.Adam(classification_model.parameters(), lr=1e-4)

In [9]:
# AttUSeg
# 594 MRI slices for training and 136 MRI slices for validation.
# This model was developed using a four-layer Attention U-Net architecture, with 16, 32, 64, and 128 features in each layer.
# The Focal Tversky Loss function, with parameters α = 0.8, β = 0.2, and γ = 1.33, was employed for training
# To mitigate overfitting, we increased the size of the validation set,
# incorporated a dropout layer with a probability of 0.2, and applied L2 regularization

segmentation_model = monai.networks.nets.AttentionUnet(
    spatial_dims=2,
    in_channels=1,
    out_channels=2,
    channels=(16, 32, 64, 128),
    strides=(2, 2, 2, 2),
    dropout=0.2,
).to(device)

model_name = "AttUnet"
loss_function = DiceLoss(to_onehot_y=True, softmax=True)
optimizer = torch.optim.Adam(segmentation_model.parameters(), lr=1e-4, weight_decay=1e-5)

In [ ]:
segmentation_model = monai.networks.nets.EfficientNetBN(
    model_name="efficientnet-b0",
    pretrained=True,
    spatial_dims=2,
    in_channels=1,
    num_classes=2,
).to(device)

model_name = "EfficientNetBN0"
loss_function = DiceLoss(to_onehot_y=True, softmax=True)
optimizer = torch.optim.Adam(segmentation_model.parameters(), lr=1e-4, weight_decay=1e-5)

Downloading: "https://github.com/lukemelas/EfficientNet-PyTorch/releases/download/1.0/efficientnet-b0-355c32eb.pth" to /root/.cache/torch/hub/checkpoints/efficientnet-b0-355c32eb.pth


100%|██████████| 20.4M/20.4M [00:00<00:00, 37.9MB/s]


In [ ]:
segmentation_model = monai.networks.nets.UNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=2,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    dropout=0.2,
).to(device)

model_name = "UNet"
loss_function = DiceLoss(to_onehot_y=True, softmax=True)
optimizer = torch.optim.Adam(segmentation_model.parameters(), lr=1e-4, weight_decay=1e-5)

# Treinamento

## Parametros de treino

In [ ]:
# Training visualization setup
%load_ext tensorboard
train_log_dir = "logs/train_data/" + datetime.now().strftime("%Y%m%d-%H%M%S")
writer = SummaryWriter(train_log_dir)

# Training parameters
max_epochs = 400
val_interval = 2
best_metric = -1
best_metric_epoch = -1

torch_save_path = '/content/drive/MyDrive/Colab Notebooks/Trained Weights/torch_best_metric' + model_name + '_' + str(max_epochs) + '.pth'

# Define metric for training
train_dice_metric = DiceMetric(include_background=True, reduction="mean", get_not_nans=False)

# Inference parameters (SlidingWindow)
roi_size = (96, 96)
sw_batch_size = 4

## Loop de treino

In [ ]:
for epoch in range(max_epochs):
    print("-------------------------")
    print(f"epoch {epoch + 1}/{max_epochs}")
    segmentation_model.train()
    epoch_loss = 0
    step = 0
    for batch_data in train_loader:
        step += 1
        inputs, labels = batch_data[0].to(device), batch_data[1].to(device)

        optimizer.zero_grad()
        outputs = segmentation_model(inputs)

        # Use the predefined loss_function (DiceLoss) which correctly handles model outputs and labels
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        print(f"{step}/{len(train_dataset) // train_loader.batch_size}, " f"train_loss: {loss.item():.4f}")

    epoch_loss /= step
    print(f"epoch {epoch + 1} average loss: {epoch_loss:.4f}")

    # Log training loss to TensorBoard
    writer.add_scalar("train_loss", epoch_loss, epoch + 1)

    if (epoch + 1) % val_interval == 0:
        segmentation_model.eval()
        with torch.no_grad():
            for val_data in val_loader:
                val_inputs, val_labels = (
                    val_data[0].to(device),
                    val_data[1].to(device),
                )

                val_outputs = sliding_window_inference(val_inputs, roi_size, sw_batch_size, segmentation_model)

                val_outputs_list = [post_trans(i) for i in decollate_batch(val_outputs)]
                val_labels_list = decollate_batch(val_labels)

                # compute metric for current iteration
                dice_metric(y_pred=val_outputs_list, y=val_labels_list)

            val_out_labels = torch.argmax(val_outputs, dim=1, keepdim=True)
            # aggregate the final mean dice result
            metric = dice_metric.aggregate().item()
            # reset the status for next validation round
            dice_metric.reset()

            # Log validation metric to TensorBoard
            writer.add_scalar("val_mean_dice", metric, epoch + 1)

            overlay = blend_images(image=val_inputs, label=val_out_labels, alpha=0.5, cmap="viridis", rescale_arrays=True)

            # Log sample images to TensorBoard
            plot_2d_or_3d_image(val_inputs, epoch + 1, writer, index=0, tag="1_image")
            plot_2d_or_3d_image(val_labels, epoch + 1, writer, index=0, tag="2_label")
            plot_2d_or_3d_image(val_outputs, epoch + 1, writer, index=0, tag="3_output")
            plot_2d_or_3d_image(val_out_labels, epoch + 1, writer, index=0, tag="4_out_label")
            plot_2d_or_3d_image(overlay, epoch + 1, writer, index=0, tag="5_overlay")

            if metric > best_metric:
                best_metric = metric
                best_metric_epoch = epoch + 1
                torch.save(segmentation_model.state_dict(), torch_save_path)
                print("saved new best metric model")

            print(
                f"current epoch: {epoch + 1} current mean dice: {metric:.4f}"
                f"\nbest mean dice: {best_metric:.4f} "
                f"at epoch: {best_metric_epoch}"
            )


print(f"train completed, best_metric: {best_metric:.4f} at epoch: {best_metric_epoch}")
writer.flush()
writer.close()

-------------------------
epoch 1/400


AssertionError: ground truth has different shape (torch.Size([1, 2, 512, 512])) from input (torch.Size([1, 2]))

## Export dos logs

In [ ]:
# Define source (current runtime logs) and destination (Drive) paths
destination_path = '/content/drive/MyDrive/TensorBoard_Logs/' + model_name + '_' + str(max_epochs) + '_' + str(best_metric)

try:
    # Copy the entire directory tree
    shutil.copytree(train_log_dir, destination_path)
    print(f"TensorBoard logs successfully saved to: {destination_path}")
except FileExistsError:
    print(f"Log directory already exists at: {destination_path}")
except Exception as e:
    print(f"Error saving logs: {e}")

TensorBoard logs successfully saved to: /content/drive/MyDrive/TensorBoard_Logs/AttUnet_400_0.5699273943901062


# Teste e Inferencia

Apos treinamento e validacao do modelo, seguimos para o teste, com outras imagens do nosso dataset que nao foram usadas anteriormente.


## Parametros de teste

Metricas especificas

In [ ]:
dice_metric = DiceMetric(include_background=True, reduction="mean", get_not_nans=False)

#metric_names=["sensitivity","specificity","precision","accuracy","balanced accuracy","f1 score"]
#conf_matrix_metrics_dict = {
#  name: ConfusionMatrixMetric(
#  include_background=True,  # Set to False if class 0 is a background class you want to ignore
#  metric_name=name,       # The derived metric you want to calculate
#  compute_sample=False,     # Set to True if you want a metric per-sample rather than across the batch
#  reduction="mean"
#) for name in metric_names}

# Usar reducao "mean_batch" pois a "mean"
sensitivity_metric = ConfusionMatrixMetric(include_background=True, metric_name="sensitivity", compute_sample=False, reduction="mean_batch")
specificity_metric = ConfusionMatrixMetric(include_background=True,metric_name="specificity",compute_sample=False,reduction="mean_batch")
precision_metric = ConfusionMatrixMetric(include_background=True,metric_name="precision",compute_sample=False,reduction="mean_batch")
accuracy_metric = ConfusionMatrixMetric(include_background=True,metric_name="accuracy",compute_sample=False,reduction="mean_batch")
balanced_accuracy_metric = ConfusionMatrixMetric(include_background=True,metric_name="balanced accuracy",compute_sample=False,reduction="mean_batch")
f1_score_metric = ConfusionMatrixMetric(include_background=True,metric_name="f1 score",compute_sample=False,reduction="mean_batch")

post_pred = AsDiscrete(argmax=True, to_onehot=2)
post_label = AsDiscrete(to_onehot=2)

# Inference parameters (SlidingWindow)
roi_size = (96, 96)
sw_batch_size = 4

## Inferencia no TestDataset

In [ ]:
segmentation_model.load_state_dict(torch.load('/content/drive/MyDrive/Colab Notebooks/Trained Weights/torch_best_metricAttUnet_400.pth', weights_only=True))
segmentation_model.eval()

dice_metric.reset()
test_inferences = list()

with torch.no_grad():
    for test_data in test_loader:

        # Move data to device
        test_images = test_data[0].to(device)
        test_labels = test_data[1].to(device)

        # Inference using the adapter
        test_outputs = sliding_window_inference(test_images, roi_size, sw_batch_size, segmentation_model)

        # Compute Metric
        # Apply post-processing to predictions
        test_pred_post = [post_trans(x) for x in decollate_batch(test_outputs)]

        # Decollate labels
        test_labels_list = decollate_batch(test_labels)

        # Calculate test metrics
        dice_metric(y_pred=test_pred_post, y=test_labels_list)

        # Processamento de valores para One-Hot, necessario para calculo de metricas baseadas em ConfusionMatrix
        val_outputs_list = [post_pred(i) for i in test_pred_post]
        val_labels_list = [post_label(i) for i in test_labels_list]

        #for metric in conf_matrix_metrics_dict.values():
        #  metric(y_pred=val_outputs_list, y = val_labels_list)

        sensitivity_metric(y_pred=val_outputs_list, y = val_labels_list)
        specificity_metric(y_pred=val_outputs_list, y = val_labels_list)
        precision_metric(y_pred=val_outputs_list, y = val_labels_list)
        accuracy_metric(y_pred=val_outputs_list, y = val_labels_list)
        balanced_accuracy_metric(y_pred=val_outputs_list, y = val_labels_list)
        f1_score_metric(y_pred=val_outputs_list, y = val_labels_list)

        test_inferences.append(test_outputs)

# Aggregate and print metrics
mean_dice_test = dice_metric.aggregate().item()

# The confusion matrix returns a list with Tensor(dict) like [tensor([0.9697], device='cuda:0')]
# the gambiarra in the print section was made to display it
sensitivity_result = sensitivity_metric.aggregate()
specificity_result = specificity_metric.aggregate()
precision_result = precision_metric.aggregate()
accuracy_result = accuracy_metric.aggregate()
balanced_accuracy_result = balanced_accuracy_metric.aggregate()
f1_score_result = f1_score_metric.aggregate()

print(f"Mean Dice: {mean_dice_test}")
#print(f"Mean Sensitivity: {sensitivity_result[0][1].cpu().detach().numpy()}")
print(f"Mean Sensitivity: {sensitivity_result[0][1].item()}")
print(f"Mean Specificity: {specificity_result[0][1].item()}")
print(f"Mean Precision: {precision_result[0][1].item()}")
print(f"Mean Accuracy: {accuracy_result[0][1].item()}")
print(f"Mean Balanced Accuracy: {balanced_accuracy_result[0][1].item()}")
print(f"Mean F1 Score: {f1_score_result[0][1].item()}")

dice_metric.reset()
sensitivity_metric.reset()
specificity_metric.reset()
precision_metric.reset()
accuracy_metric.reset()
balanced_accuracy_metric.reset()
f1_score_metric.reset()

#for name, metric in conf_matrix_metrics_dict.items():
#  # Get the aggregated value. ConfusionMatrixMetric aggregate returns a list of tensors.
#  agg_res = metric.aggregate()
#  if isinstance(agg_res, list) or isinstance(agg_res, tuple):
#      metric_result = agg_res[0].item()
#  else:
#      metric_result = agg_res.item()
#
#  print(f"{name.capitalize()}: {metric_result:.4f}")
#
#  metric.reset()


Mean Dice: 0.5583773255348206
Mean Sensitivity: 0.9697145819664001
Mean Specificity: 0.9697145819664001
Mean Precision: 0.9697145819664001
Mean Accuracy: 0.9697145819664001
Mean Balanced Accuracy: 0.9697145819664001
Mean F1 Score: 0.9697145819664001


In [ ]:
print(sensitivity_result)

[tensor([0.9697], device='cuda:0')]


## Visualizacao das metricas e resultados

In [ ]:
# @title
## FOR 3D DATA
og_test_images = list()
og_test_labels = list()

for test_data in test_loader:
  og_test_images.append(test_data[0])
  og_test_labels.append(test_data[1])

def interactive_test_image_visualization(img_no, img_slice):
  # Convert tensors to numpy arrays for visualization
  # Input Shape: (Batch, Channel, Height, Width, Depth)
  original_image = og_test_images[img_no][0, 0, :, :, img_slice].cpu().detach().numpy()
  original_label = og_test_labels[img_no][0, 0, :, :, img_slice].cpu().detach().numpy()

  # Output Shape: (Batch, Classes, Height, Width, Depth)
  original_output_logits = test_inferences[img_no]
  # Get the predicted class (0 or 1) by argmax
  original_output = torch.argmax(original_output_logits, dim=1)[0, :, :, img_slice].cpu().detach().numpy()

  # Create masked arrays for overlay visualization (0 values are transparent)
  masked_label = np.ma.masked_where(original_label == 0, original_label)
  masked_output = np.ma.masked_where(original_output == 0, original_output)

  plt.figure("check", (24, 6))
  plt.subplot(1, 4, 1)
  plt.title(f"Original Image")
  plt.imshow(original_image, cmap='gray')
  plt.subplot(1, 4, 2)
  plt.title(f"Ground Truth")
  plt.imshow(original_label, cmap='gray')
  plt.subplot(1, 4, 3)
  plt.title(f"Model Prediction")
  plt.imshow(original_output, cmap='gray')
  plt.subplot(1, 4, 4)
  plt.title(f"Overlay (GT: Green, Pred: Red)")
  plt.imshow(original_image, cmap='gray')
  # Ground Truth in Green
  plt.imshow(masked_label, cmap='Greens', alpha=0.8, vmin=0, vmax=1)
  # Prediction in Red
  plt.imshow(masked_output, cmap='Reds', alpha=0.8, vmin=0, vmax=1)
  plt.show()


interactive(interactive_test_image_visualization, img_no=(0,len(og_test_images)-1), img_slice=(0,63))

In [ ]:
## FOR 2D DATA
og_test_images = list()
og_test_labels = list()

for test_data in test_loader:
  og_test_images.append(test_data[0])
  og_test_labels.append(test_data[1])

def interactive_test_image_visualization(img_no):
  # Convert tensors to numpy arrays for visualization
  # Input Shape: (Batch, Channel, Height, Width)
  original_image = og_test_images[img_no][0, 0, :, :].cpu().detach().numpy()
  original_label = og_test_labels[img_no][0, 0, :, :].cpu().detach().numpy()

  # Output Shape: (Batch, Classes, Height, Width)
  original_output_logits = test_inferences[img_no]
  # Get the predicted class (0 or 1) by argmax
  original_output = torch.argmax(original_output_logits, dim=1)[0, :, :].cpu().detach().numpy()

  # Apply post-processing properly for the single inference
  # Add batch dimension back to pass into get_confusion_matrix
  cnf_outputs = post_pred(test_inferences[img_no][0]).unsqueeze(0)
  cnf_labels = post_label(og_test_labels[img_no][0]).unsqueeze(0)
  cnf_matrix = get_confusion_matrix(cnf_outputs, cnf_labels)

  # Extract TP, FP, TN, FN for the foreground class (class 1)
  # cnf_matrix shape is [1, 2, 4] (batch, class, values)
  tp, fp, tn, fn = cnf_matrix[0, 1].cpu().detach().numpy()

  # Create masked arrays for overlay visualization (0 values are transparent)
  masked_label = np.ma.masked_where(original_label == 0, original_label)
  masked_output = np.ma.masked_where(original_output == 0, original_output)

  plt.figure("check", (30, 6))
  plt.subplot(1, 5, 1)
  plt.title(f"Original Image")
  plt.imshow(original_image, cmap='gray')
  plt.subplot(1, 5, 2)
  plt.title(f"Ground Truth")
  plt.imshow(original_label, cmap='gray')
  plt.subplot(1, 5, 3)
  plt.title(f"Model Prediction")
  plt.imshow(original_output, cmap='gray')
  plt.subplot(1, 5, 4)
  plt.title(f"Overlay (GT: Green, Pred: Red)")
  plt.imshow(original_image, cmap='gray')
  # Ground Truth in Green
  plt.imshow(masked_label, cmap='Greens', alpha=0.8, vmin=0, vmax=1)
  # Prediction in Red
  plt.imshow(masked_output, cmap='Reds', alpha=0.8, vmin=0, vmax=1)

  plt.subplot(1, 5, 5)
  plt.axis('off')
  plt.title("Confusion Matrix (Foreground)")
  table_data = [
      ["", "Predicted Positive", "Predicted Negative"],
      ["Actual Positive", f"{int(tp)}", f"{int(fn)}"],
      ["Actual Negative", f"{int(fp)}", f"{int(tn)}"]
  ]
  table = plt.table(cellText=table_data, loc='center', cellLoc='center')
  table.auto_set_font_size(False)
  table.set_fontsize(14)
  table.scale(1.2, 2.5)
  plt.show()

interactive(interactive_test_image_visualization, img_no=(0,len(og_test_images)-1))

interactive(children=(IntSlider(value=3, description='img_no', max=6), Output()), _dom_classes=('widget-intera…